In [2]:
from parse_aquisition_time import rename_sites, parse_acquisition_timestamps_with_rewells
import pandas as pd
import numpy as np

In [3]:
tps0 = rename_sites(parse_acquisition_timestamps_with_rewells([r"M:\marvwy\RESTORED\cycle0", r"M:\marvwy\RESTORED\cycle0_reimage"]))
tps1 = rename_sites(parse_acquisition_timestamps_with_rewells([r"M:\marvwy\RESTORED\cycle1"]))
tps2 = rename_sites(parse_acquisition_timestamps_with_rewells([r"M:\marvwy\RESTORED\cycle2", r"M:\marvwy\RESTORED\cycle2_reimage", r"M:\marvwy\RESTORED\cycle2_reimage2"]))
tps3 = rename_sites(parse_acquisition_timestamps_with_rewells([r"M:\marvwy\RESTORED\cycle3", r"M:\marvwy\RESTORED\cycle3_reimage"]))

In [9]:
tall_dfs = []

for i, e in enumerate([tps0, tps1, tps2, tps3]):
    tps0_tall = pd.DataFrame(e).T.stack().rename('acquisitionTime').to_frame().reset_index()
    tps0_tall.index = tps0_tall[['level_0', 'level_1']].agg('_'.join, axis=1).rename('roi')
    tps0_tall['channel'] = tps0_tall['level_2']
    # tps0_tall['timeDeltaMinutes'] = (tps0_tall['acquisitionTime'] - tps0_tall['acquisitionTime'].min()).apply(pd.Timedelta.total_seconds).apply(lambda x: x / 60)
    tps0_tall['timeDelta'] = (tps0_tall['acquisitionTime'] - tps0_tall['acquisitionTime'].min()).apply(pd.Timedelta)
    tps0_tall['timeDeltaMinutes'] = (tps0_tall['timeDelta'].dt.total_seconds() / 60).astype(pd.Float32Dtype())
    tps0_tall.drop(['level_0', 'level_1', 'level_2'], inplace=True, axis=1)
    tps0_tall = tps0_tall[tps0_tall.channel=='1'].drop('channel', axis=1).assign(acquisition=i)
    tall_dfs.append(tps0_tall.reset_index())

tps = pd.concat(tall_dfs, axis=0)
tps = tps.astype({'roi': pd.StringDtype(), 'acquisition': pd.Int64Dtype()})
# tps['acquisition'] = tps['acquisition'].astype(np.uint8)
tps = tps.set_index(['roi', 'acquisition']).reset_index()

In [10]:
tps

,roi,acquisition,acquisitionTime,timeDelta,timeDeltaMinutes
0,B02_px-2133_py+0159,0,2022-07-21 18:50:02.934000+02:00,0 days 00:09:10.006000,9.166767
1,B02_px-1488_py-1765,0,2022-07-21 18:56:08.563000+02:00,0 days 00:15:15.635000,15.260583
2,B02_px-1307_py+2200,0,2022-07-21 18:43:56.523000+02:00,0 days 00:03:03.595000,3.059917
3,B02_px+0198_py-2152,0,2022-07-21 18:59:11.918000+02:00,0 days 00:18:18.990000,18.3165
4,B02_px+0385_py-0060,0,2022-07-21 18:53:05.436000+02:00,0 days 00:12:12.508000,12.208467
...,...,...,...,...,...
843,G07_px+0166_py+1561,3,2022-07-29 13:17:20.032000+02:00,0 days 16:06:14.863000,966.247742
844,G07_px+0320_py-1306,3,2022-07-29 13:33:16.896000+02:00,0 days 16:22:11.727000,982.195435
845,G07_px+1302_py-1520,3,2022-07-29 13:36:28.305000+02:00,0 days 16:25:23.136000,985.38562
846,G07_px+1541_py+1412,3,2022-07-29 13:20:31.114000+02:00,0 days 16:09:25.945000,969.432434


In [7]:
tps.to_parquet(r"C:\Users\hessm\Documents\Programming\Python\zfish\data\acquisition_times.parquet")